In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hedis.gold;

In [0]:
%sql
-- ============================================================
-- BCS — Breast Cancer Screening
-- ============================================================
CREATE OR REPLACE TABLE hedis.gold.bcs_measure AS
WITH eligible AS (
  SELECT DISTINCT p.patient_id, p.region, p.race, p.ethnicity
  FROM hedis.silver.dim_patient p
  JOIN hedis.gold.continuous_enrollment_2025 ce ON p.patient_id = ce.patient_id
  WHERE p.gender = 'F'
    AND datediff('2025-12-31', p.birthdate)/365.25 BETWEEN 50 AND 74
    AND (p.deathdate IS NULL OR p.deathdate > '2025-12-31')
),
numerator AS (
  SELECT DISTINCT patient_id
  FROM hedis.silver.fact_procedure
  WHERE procedure_code IN (71651007, 241055006, 24623002)
    AND procedure_date BETWEEN '2023-10-01' AND '2025-12-31'  -- 27-month lookback
)
SELECT
  el.patient_id, el.region, el.race, el.ethnicity,
  2025 AS measurement_year,
  CASE WHEN n.patient_id IS NOT NULL THEN 1 ELSE 0 END AS compliant
FROM eligible el
LEFT JOIN numerator n ON el.patient_id = n.patient_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- ============================================================
-- COL — Colorectal Cancer Screening (colonoscopy only)
-- ============================================================
CREATE OR REPLACE TABLE hedis.gold.col_measure AS
WITH eligible AS (
  SELECT DISTINCT p.patient_id, p.region, p.race, p.ethnicity
  FROM hedis.silver.dim_patient p
  JOIN hedis.gold.continuous_enrollment_2025 ce ON p.patient_id = ce.patient_id
  WHERE datediff('2025-12-31', p.birthdate)/365.25 BETWEEN 45 AND 75
    AND (p.deathdate IS NULL OR p.deathdate > '2025-12-31')
),
numerator AS (
  SELECT DISTINCT patient_id
  FROM hedis.silver.fact_procedure
  WHERE procedure_code = 73761001
    AND procedure_date BETWEEN '2015-12-31' AND '2025-12-31'  -- 10-year lookback
)
SELECT
  el.patient_id, el.region, el.race, el.ethnicity,
  2025 AS measurement_year,
  CASE WHEN n.patient_id IS NOT NULL THEN 1 ELSE 0 END AS compliant
FROM eligible el
LEFT JOIN numerator n ON el.patient_id = n.patient_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- ============================================================
-- CDC — Comprehensive Diabetes Care (HbA1c Testing)
-- ============================================================
CREATE OR REPLACE TABLE hedis.gold.cdc_hba1c_measure AS
WITH diabetics AS (
  SELECT DISTINCT patient_id
  FROM hedis.silver.fact_condition
  WHERE condition_code = 44054006
    AND start_date <= '2025-12-31'
    AND (stop_date IS NULL OR stop_date > '2025-01-01')
),
eligible AS (
  SELECT DISTINCT p.patient_id, p.region, p.race, p.ethnicity
  FROM hedis.silver.dim_patient p
  JOIN diabetics d ON p.patient_id = d.patient_id
  JOIN hedis.gold.continuous_enrollment_2025 ce ON p.patient_id = ce.patient_id
  WHERE (p.deathdate IS NULL OR p.deathdate > '2025-12-31')
),
numerator AS (
  SELECT DISTINCT patient_id
  FROM hedis.silver.fact_observation
  WHERE obs_code = '4548-4'
    AND obs_date BETWEEN '2025-01-01' AND '2025-12-31'
)
SELECT
  el.patient_id, el.region, el.race, el.ethnicity,
  2025 AS measurement_year,
  CASE WHEN n.patient_id IS NOT NULL THEN 1 ELSE 0 END AS compliant
FROM eligible el
LEFT JOIN numerator n ON el.patient_id = n.patient_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- ============================================================
-- CBP — Controlling High Blood Pressure
-- ============================================================
CREATE OR REPLACE TABLE hedis.gold.cbp_measure AS
WITH hypertensives AS (
  SELECT DISTINCT patient_id
  FROM hedis.silver.fact_condition
  WHERE condition_code = 59621000
    AND start_date <= '2025-12-31'
    AND (stop_date IS NULL OR stop_date > '2025-01-01')
),
eligible AS (
  SELECT DISTINCT p.patient_id, p.region, p.race, p.ethnicity
  FROM hedis.silver.dim_patient p
  JOIN hypertensives h ON p.patient_id = h.patient_id
  JOIN hedis.gold.continuous_enrollment_2025 ce ON p.patient_id = ce.patient_id
  WHERE datediff('2025-12-31', p.birthdate)/365.25 >= 18
    AND (p.deathdate IS NULL OR p.deathdate > '2025-12-31')
),
latest_bp AS (
  SELECT patient_id, systolic, diastolic,
         ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY obs_date DESC) AS rn
  FROM hedis.silver.fact_vitals_bp
  WHERE obs_date BETWEEN '2025-01-01' AND '2025-12-31'
)
SELECT
  el.patient_id, el.region, el.race, el.ethnicity,
  2025 AS measurement_year,
  CASE WHEN bp.systolic < 140 AND bp.diastolic < 90 THEN 1 ELSE 0 END AS compliant
FROM eligible el
LEFT JOIN latest_bp bp ON el.patient_id = bp.patient_id AND bp.rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE hedis.gold.measure_summary AS
WITH all_measures AS (
  SELECT 'BCS' AS measure, region, race, ethnicity, compliant FROM hedis.gold.bcs_measure
  UNION ALL
  SELECT 'COL' AS measure, region, race, ethnicity, compliant FROM hedis.gold.col_measure
  UNION ALL
  SELECT 'CDC-HbA1c' AS measure, region, race, ethnicity, compliant FROM hedis.gold.cdc_hba1c_measure
  UNION ALL
  SELECT 'CBP' AS measure, region, race, ethnicity, compliant FROM hedis.gold.cbp_measure WHERE compliant IS NOT NULL
),
by_region AS (
  SELECT measure, 'region' AS slice_type, region AS slice_value, COUNT(*) AS denominator, SUM(compliant) AS numerator
  FROM all_measures GROUP BY measure, region
),
by_race AS (
  SELECT measure, 'race' AS slice_type, race AS slice_value, COUNT(*) AS denominator, SUM(compliant) AS numerator
  FROM all_measures GROUP BY measure, race
),
by_ethnicity AS (
  SELECT measure, 'ethnicity' AS slice_type, ethnicity AS slice_value, COUNT(*) AS denominator, SUM(compliant) AS numerator
  FROM all_measures GROUP BY measure, ethnicity
)
SELECT *,
  ROUND(numerator / denominator * 100, 1) AS compliance_rate,
  CASE WHEN denominator < 30 THEN true ELSE false END AS suppressed
FROM (SELECT * FROM by_region UNION ALL SELECT * FROM by_race UNION ALL SELECT * FROM by_ethnicity)
ORDER BY measure, slice_type, slice_value;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE hedis.gold.measure_summary_overall AS
WITH all_measures AS (
  SELECT 'Breast cancer screening' AS measure, compliant FROM hedis.gold.bcs_measure
  UNION ALL
  SELECT 'Colorectal screening' AS measure, compliant FROM hedis.gold.col_measure
  UNION ALL
  SELECT 'Diabetes care — HbA1c' AS measure, compliant FROM hedis.gold.cdc_hba1c_measure
  UNION ALL
  SELECT 'Blood pressure control' AS measure, compliant FROM hedis.gold.cbp_measure WHERE compliant IS NOT NULL
)
SELECT measure,
  COUNT(*) AS eligible_count,
  SUM(compliant) AS numerator,
  ROUND(SUM(compliant) / COUNT(*) * 100, 1) AS compliance_rate
FROM all_measures
GROUP BY measure
ORDER BY measure;

num_affected_rows,num_inserted_rows


In [0]:
from pyspark.sql.functions import col, datediff, to_date, lit
patients = spark.table("hedis.bronze.patients")

df = patients.groupBy("STATE").count().orderBy(col("count").desc())
df.write.mode("overwrite").saveAsTable("hedis.gold.metrics")

In [0]:
%sql
SELECT measure, slice_type, slice_value, denominator, compliance_rate, suppressed
FROM hedis.gold.measure_summary
ORDER BY measure, slice_type, suppressed DESC;

measure,slice_type,slice_value,denominator,compliance_rate,suppressed
BCS,race,hawaiian,20,5.0,true
BCS,race,native,17,0.0,true
BCS,race,other,17,11.8,true
BCS,race,white,1156,4.5,false
BCS,race,asian,106,4.7,false
BCS,race,black,185,7.0,false
BCS,region,South,849,4.2,false
BCS,region,West,256,6.6,false
BCS,region,Northeast,396,5.1,false
CBP,race,native,26,73.1,true


In [0]:
%sql
-- how many BCS-eligible women have a qualifying mammogram EVER, not just in the 27-month window?
SELECT COUNT(DISTINCT el.patient_id) AS lifetime_any_mammogram
FROM hedis.gold.bcs_measure el
JOIN hedis.silver.fact_procedure p
  ON el.patient_id = p.patient_id
  AND p.procedure_code IN (71651007, 241055006, 24623002)

lifetime_any_mammogram
76
